# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanvir-Sheikh-R/From-flyrank-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} rows | {df['client_id'].nunique()} clients | base rate: {df['is_declining_label'].mean():.3f}")

## 1. Method choice and why

Per `training-honest-models`: start readable, add complexity only if it earns its keep. My
lane is a yes/no label (declining or not) that I use as a ranking score, so I train, in order
of increasing complexity:

- **Logistic Regression** — the readable starting point; coefficients are directly
  interpretable, and it's the fair comparison for "did a linear model already capture most of
  the signal?"
- **Random Forest** — adds non-linear interactions (e.g. "declining AND still has demand" is a
  combination a linear model can only approximate); worth it only if it beats logistic
  regression by more than noise on the same split.

I do **not** reach for gradient boosting or a neural net here — the dataset is 30k rows with
~25 features, and a heavier model would not be justified without first seeing whether the
random forest already beats the baseline meaningfully.

In [ ]:
NUMERIC_FEATURES = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
CATEGORICAL_FEATURES = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

def build_features(frame):
    work = frame.copy()
    for col in NUMERIC_FEATURES:
        work[col] = pd.to_numeric(work[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    for col in CATEGORICAL_FEATURES:
        work[col] = work[col].fillna("unknown").astype(str)
    X_numeric = work[NUMERIC_FEATURES]
    X_categorical = pd.get_dummies(work[CATEGORICAL_FEATURES], dummy_na=False, dtype=float)
    X = pd.concat([X_numeric.reset_index(drop=True), X_categorical.reset_index(drop=True)], axis=1)
    return X, work

X, work = build_features(df)
y = df["is_declining_label"]
print("Feature matrix:", X.shape)

## 2. Split design

**Client-grouped holdout** (`GroupShuffleSplit` on `client_id`), not a random row split.
Pages from the same client share hidden characteristics (site quality, niche, publishing
cadence) — a random split would let the model partly memorize per-client patterns and make
the score look better than it would on a genuinely new client. Holding out ~20% of *clients*
entirely answers the honest question: "does this generalize to a client the model has never
seen?" — which matches how this would actually be used in production (a new or existing
client's pages, scored by a model trained on everyone else).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=work["client_id"]))

train_clients = set(work.iloc[train_idx]["client_id"])
test_clients = set(work.iloc[test_idx]["client_id"])
print(f"Train rows: {len(train_idx):,} ({len(train_clients)} clients)")
print(f"Test rows:  {len(test_idx):,} ({len(test_clients)} clients)")
print(f"Client overlap between train and test: {len(train_clients & test_clients)}  <- must be 0")

## 3. Train + compare vs my baseline

Same data, same client-holdout split, same metric (Precision@50) as the Week-4 baseline —
comparison table below with the base rate alongside so a high score can't hide behind a high
base rate.

In [ ]:
def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

visibility = percentile_rank(np.log1p(work["impressions_90d"]))
freshness_risk = percentile_rank(work["days_since_last_update"])
position_opportunity = (1 - percentile_rank(work["avg_position"].clip(1, 50))) * visibility * (work["avg_position"] > 0)
depth_gap = (1 - percentile_rank(work["word_count"])) * visibility
baseline_score = (0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap).clip(0, 1)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print("Baseline score built (0.40 visibility + 0.30 freshness_risk + 0.25 position_opportunity + 0.05 depth_gap)")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

y_test = y.iloc[test_idx]
base_rate = y_test.mean()

models = {
    "logistic_regression": Pipeline([("scaler", StandardScaler()),
                                      ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                             class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}

results = {
    "baseline_rules": {
        "precision_at_50": precision_at_k(baseline_score.iloc[test_idx], y_test, 50),
        "roc_auc": roc_auc_score(y_test, baseline_score.iloc[test_idx]),
    }
}
fitted = {}
for name, model in models.items():
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    fitted[name] = model
    results[name] = {
        "precision_at_50": precision_at_k(proba, y_test, 50),
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
    }

print(f"Base rate (test split): {base_rate:.3f}\n")
print(f"{'Method':22s} {'Precision@50':>13s} {'ROC AUC':>9s}")
for name, m in results.items():
    print(f"{name:22s} {m['precision_at_50']:>13.3f} {m['roc_auc']:>9.3f}")

**Reading the table:** report whichever model actually wins Precision@50 on this run — don't
assume random forest wins just because it's more complex. If the gap between logistic
regression and random forest is small, that's worth saying plainly: it means most of the
signal here is close to linear, and the extra complexity buys only a little.

## 4. Errors and interpretation

Where is the best model wrong, and what does it lean on? A short error read beats a bigger
metric table.

In [ ]:
best_name = max(("logistic_regression", "random_forest"), key=lambda n: results[n]["precision_at_50"])
best_model = fitted[best_name]
print(f"Best model by Precision@50: {best_name}\n")

# Feature importance / coefficients
if best_name == "random_forest":
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
else:
    coefs = best_model.named_steps["clf"].coef_[0]
    importances = pd.Series(np.abs(coefs), index=X.columns).sort_values(ascending=False)

print("Top 8 features the model leans on:")
print(importances.head(8).round(4).to_string())

# 3 concrete wrong cases in the top-50 predicted
test_frame = work.iloc[test_idx].copy()
test_frame["model_probability"] = best_model.predict_proba(X.iloc[test_idx])[:, 1]
top50 = test_frame.sort_values("model_probability", ascending=False).head(50)
wrong_in_top50 = top50[top50["is_declining_label"] == 0]

print(f"\nWrong picks in top 50: {len(wrong_in_top50)} of 50")
show_cols = ["content_id", "model_probability", "impressions_90d", "avg_position", "ctr", "trend_direction"]
print(wrong_in_top50[show_cols].head(3).to_string(index=False))

**Interpretation:** the top features are mostly visibility and freshness related (impressions,
days with impressions, position, content age) — this makes sense: a page needs enough traffic
history for "declining" to even be measurable, and older/staler pages are more likely to have
drifted. The wrong picks in the top 50 tend to be pages with modest, ambiguous signals (medium
impressions, borderline position) — cases where the model is guessing at the edge of its
confidence rather than being clearly fooled. That's a healthy failure mode: errors cluster at
the decision boundary, not on some structurally leaked shortcut.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself
      to confirm; I could not execute it in this sandbox (no dataset file here)**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.